<a href="https://colab.research.google.com/github/hkr-tello/ai-hero/blob/main/vscode_docs_days1_to_3.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 🤖 VS Code Docs AI Agent — Days 1, 2 & 3
**Dataset:** `microsoft/vscode-docs` (~780 markdown docs)

| Day | Topic |
|-----|-------|
| 1   | Download & inspect the data |
| 2   | Chunk it two ways and compare |
| 3   | Build hybrid search (text + vector) |

---
# 📥 DAY 1 — Download & Inspect vscode-docs

In [1]:
# Install dependencies
!pip install requests python-frontmatter -q

In [2]:
import io
import zipfile
import requests
import frontmatter

def read_repo_data(repo_owner, repo_name):
    """Download and parse all .md and .mdx files from a GitHub repo."""
    url = f'https://codeload.github.com/{repo_owner}/{repo_name}/zip/refs/heads/main'
    print(f'Downloading {repo_owner}/{repo_name}...')
    resp = requests.get(url)

    if resp.status_code != 200:
        raise Exception(f'Failed to download: HTTP {resp.status_code}')

    repository_data = []
    zf = zipfile.ZipFile(io.BytesIO(resp.content))

    for file_info in zf.infolist():
        filename = file_info.filename
        filename_lower = filename.lower()

        if not (filename_lower.endswith('.md') or filename_lower.endswith('.mdx')):
            continue

        try:
            with zf.open(file_info) as f_in:
                content = f_in.read().decode('utf-8', errors='ignore')
                post = frontmatter.loads(content)
                data = post.to_dict()
                data['filename'] = filename
                repository_data.append(data)
        except Exception as e:
            print(f'  Skipped {filename}: {e}')
            continue

    zf.close()
    print(f'Done. {len(repository_data)} files loaded.')
    return repository_data

In [3]:
# Download vscode-docs from GitHub
vscode_docs = read_repo_data('microsoft', 'vscode-docs')

Done. 782 files loaded.


In [4]:
# Inspect the structure of a document
sample = vscode_docs[10]
print('Available fields:', list(sample.keys()))
print()
print('Title   :', sample.get('title', 'N/A'))
print('Filename:', sample.get('filename', 'N/A'))
print()
print('Content preview (first 500 chars):')
print(sample.get('content', '')[:500])

Available fields: ['description', 'argument-hint', 'tools', 'content', 'filename']

Title   : N/A
Filename: vscode-docs-main/.github/prompts/review-blog.prompt.md

Content preview (first 500 chars):
Review the article for clarity, conciseness, and adherence to our blog post [writing style guidelines](../instructions/blog-writing.instructions.md).

Provide concrete and practical suggestions for improvement.


In [5]:
# Analyze content size distribution
import statistics

lengths = [len(str(d.get('content', ''))) for d in vscode_docs]

print('=== CONTENT SIZE ANALYSIS ===')
print(f'Total docs         : {len(vscode_docs)}')
print(f'Average length     : {int(statistics.mean(lengths)):,} chars')
print(f'Median length      : {int(statistics.median(lengths)):,} chars')
print(f'Max length         : {max(lengths):,} chars')
print(f'Min length         : {min(lengths):,} chars')
print(f'Docs > 2,000 chars : {sum(1 for l in lengths if l > 2000)} (need chunking)')
print(f'Docs < 100 chars   : {sum(1 for l in lengths if l < 100)} (too small, will filter)')

=== CONTENT SIZE ANALYSIS ===
Total docs         : 782
Average length     : 15,483 chars
Median length      : 8,457 chars
Max length         : 99,920 chars
Min length         : 56 chars
Docs > 2,000 chars : 692 (need chunking)
Docs < 100 chars   : 3 (too small, will filter)


In [6]:
# Filter out empty or near-empty documents
vscode_docs = [d for d in vscode_docs if len(str(d.get('content', ''))) > 100]
print(f'Docs after filtering empties: {len(vscode_docs)}')
print('Day 1 complete! Data is ready for chunking.')

Docs after filtering empties: 779
Day 1 complete! Data is ready for chunking.


---
# ✂️ DAY 2 — Chunking: Compare Both Strategies

## Strategy A: Sliding Window (character-based with overlap)

In [7]:
def sliding_window(seq, size=2000, step=1000):
    """Split text into overlapping fixed-size chunks."""
    if size <= 0 or step <= 0:
        raise ValueError('size and step must be positive')
    n = len(seq)
    result = []
    for i in range(0, n, step):
        chunk = seq[i:i + size]
        result.append({'start': i, 'chunk': chunk})
        if i + size >= n:
            break
    return result


# Apply to all vscode docs
vscode_sw_chunks = []

for doc in vscode_docs:
    doc_copy = doc.copy()
    doc_content = doc_copy.pop('content')
    chunks = sliding_window(doc_content, size=2000, step=1000)
    for chunk in chunks:
        chunk.update(doc_copy)
    vscode_sw_chunks.extend(chunks)

print(f'Sliding window total chunks: {len(vscode_sw_chunks)}')
print()
print('Sample chunk:')
print(vscode_sw_chunks[0]['chunk'][:400])

Sliding window total chunks: 11770

Sample chunk:
Please review the provided code changes or pull request details for quality, style, and adherence to best practices. Provide constructive feedback and suggest improvements where necessary.


## Strategy B: Section-Based (split by ## markdown headers)

In [8]:
import re

def split_markdown_by_level(text, level=2):
    """Split markdown text by header level (default: ## H2 headers)."""
    header_pattern = r'^(#{' + str(level) + r'} )(.+)$'
    pattern = re.compile(header_pattern, re.MULTILINE)
    parts = pattern.split(text)

    sections = []
    for i in range(1, len(parts), 3):
        header = (parts[i] + parts[i + 1]).strip()
        content = parts[i + 2].strip() if i + 2 < len(parts) else ''
        section = f'{header}\n\n{content}' if content else header
        sections.append(section)

    return sections


# Apply to all vscode docs
vscode_section_chunks = []

for doc in vscode_docs:
    doc_copy = doc.copy()
    doc_content = doc_copy.pop('content')
    sections = split_markdown_by_level(doc_content, level=2)

    if not sections:
        # Doc has no ## headers — keep the whole content as one section
        doc_copy['section'] = doc_content.strip()
        vscode_section_chunks.append(doc_copy)
    else:
        for section in sections:
            section_doc = doc_copy.copy()
            section_doc['section'] = section
            vscode_section_chunks.append(section_doc)

print(f'Section-based total chunks: {len(vscode_section_chunks)}')
print()
print('Sample section chunk:')
print(vscode_section_chunks[5]['section'][:400])

Section-based total chunks: 5735

Sample section chunk:
## Writing style

- In general, the blog post should adhere to the docs [writing guidelines](./docs-writing.instructions.md).
- Blog posts should be engaging and can include a more conversational tone than standard documentation.
- Use active voice and first-person plural ("we") to create a sense of community and shared experience.
- Use contractions (e.g., "it's", "we're") to make the writing fee


## Head-to-Head Comparison

In [9]:
import statistics

sw_sizes  = [len(c['chunk'])   for c in vscode_sw_chunks]
sec_sizes = [len(c['section']) for c in vscode_section_chunks]

print('┌─────────────────────────┬──────────────────┬──────────────────┐')
print('│ Metric                  │ Sliding Window   │ Section-Based    │')
print('├─────────────────────────┼──────────────────┼──────────────────┤')
print(f'│ Total chunks            │ {len(vscode_sw_chunks):<16} │ {len(vscode_section_chunks):<16} │')
print(f'│ Avg chunk size (chars)  │ {int(statistics.mean(sw_sizes)):<16,} │ {int(statistics.mean(sec_sizes)):<16,} │')
print(f'│ Max chunk size (chars)  │ {max(sw_sizes):<16,} │ {max(sec_sizes):<16,} │')
print(f'│ Min chunk size (chars)  │ {min(sw_sizes):<16,} │ {min(sec_sizes):<16,} │')
print('├─────────────────────────┼──────────────────┼──────────────────┤')
print('│ Cuts mid-sentence?      │ Yes              │ No               │')
print('│ Semantically complete?  │ Sometimes        │ Yes              │')
print('│ Uniform size?           │ Yes              │ No               │')
print('└─────────────────────────┴──────────────────┴──────────────────┘')
print()
print('✅ WINNER for vscode-docs: Section-based')
print('   Reason: vscode-docs is well-structured technical markdown.')
print('   Section-based gives semantically complete, self-contained chunks.')

┌─────────────────────────┬──────────────────┬──────────────────┐
│ Metric                  │ Sliding Window   │ Section-Based    │
├─────────────────────────┼──────────────────┼──────────────────┤
│ Total chunks            │ 11770            │ 5735             │
│ Avg chunk size (chars)  │ 1,962            │ 1,987            │
│ Max chunk size (chars)  │ 2,000            │ 58,195           │
│ Min chunk size (chars)  │ 133              │ 6                │
├─────────────────────────┼──────────────────┼──────────────────┤
│ Cuts mid-sentence?      │ Yes              │ No               │
│ Semantically complete?  │ Sometimes        │ Yes              │
│ Uniform size?           │ Yes              │ No               │
└─────────────────────────┴──────────────────┴──────────────────┘

✅ WINNER for vscode-docs: Section-based
   Reason: vscode-docs is well-structured technical markdown.
   Section-based gives semantically complete, self-contained chunks.


In [10]:
# Manually inspect 3 section chunks to confirm quality
for i in [0, 10, 50]:
    c = vscode_section_chunks[i]
    print(f'--- Chunk {i} ---')
    print(f'Title    : {c.get("title", "N/A")}')
    print(f'File     : {c.get("filename", "").split("/")[-1]}')
    print(f'Size     : {len(c["section"])} chars')
    print(f'Content  : {c["section"][:250]}')
    print()

--- Chunk 0 ---
Title    : N/A
File     : code-reviewer.agent.md
Size     : 188 chars
Content  : Please review the provided code changes or pull request details for quality, style, and adherence to best practices. Provide constructive feedback and suggest improvements where necessary.

--- Chunk 10 ---
Title    : N/A
File     : docs-writing.instructions.md
Size     : 555 chars
Content  : ## Punctuation

* Use short, simple sentences.
* End all sentences with a period.
* Use one space after punctuation marks.
* After a colon, capitalize only proper nouns.
* Avoid semicolons - use separate sentences instead.
* Avoid em-dashes and prefe

--- Chunk 50 ---
Title    : N/A
File     : SKILL.md
Size     : 1372 chars
Content  : ## Stable Release Notes

Stable release notes summarize the key features and improvements in a stable release of VS Code. They follow a more structured format with predefined sections for different feature areas. The release is intially created using



In [11]:
# Lock in section-based as final chunks for Day 3
final_chunks = vscode_section_chunks
print(f'Final chunk count ready for indexing: {len(final_chunks)}')
print(f'Fields per chunk: {list(final_chunks[0].keys())}')
print('Day 2 complete!')

Final chunk count ready for indexing: 5735
Fields per chunk: ['name', 'description', 'argument-hint', 'tools', 'filename', 'section']
Day 2 complete!


---
# 🔍 DAY 3 — Hybrid Search (Text + Vector)

In [12]:
!pip install minsearch sentence-transformers -q

## Part 1: Text Search (Lexical / Keyword)

In [13]:
from minsearch import Index

text_index = Index(
    text_fields=['section', 'title', 'filename'],
    keyword_fields=[]
)
text_index.fit(final_chunks)
print(f'Text index built — {len(final_chunks)} chunks indexed')

Text index built — 5735 chunks indexed


In [14]:
# Test text search
query = 'How do I configure keyboard shortcuts in VS Code?'
text_results = text_index.search(query, num_results=3)

print(f'Query: "{query}"')
print(f'Text search results: {len(text_results)}')
print()
for i, r in enumerate(text_results):
    print(f'[{i+1}] {r.get("title", "No title")} | {r.get("filename", "").split("/")[-1]}')
    print(f'     {r["section"][:200]}')
    print()

Query: "How do I configure keyboard shortcuts in VS Code?"
Text search results: 3

[1] No title | keybindings.md
     ## Keyboard Shortcuts editor

VS Code provides a rich keyboard shortcut editing experience with the Keyboard Shortcuts editor. The editor lists all available commands with and without keyboard shortcu

[2] No title | keybindings.md
     ## Viewing modified keyboard shortcuts

To filter the list to only show the shortcuts you have modified, select the **Show User Keybindings** command in the **More Actions** (**...**) menu. This appli

[3] No title | keybindings.md
     ## Advanced customization

VS Code keeps track of the keyboard shortcuts you have customized in the `keybindings.json` file. For advanced customization, you can also directly modify the `keybindings.j



## Part 2: Vector Search (Semantic / Embedding-based)

In [15]:
from sentence_transformers import SentenceTransformer

# Downloads ~90MB on first run — optimized for Q&A tasks
embedding_model = SentenceTransformer('multi-qa-distilbert-cos-v1')
print('Embedding model loaded!')

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/333 [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Embedding model loaded!


In [16]:
from tqdm.auto import tqdm
import numpy as np

# Build embeddings for all chunks
# ~780 section chunks takes ~2-3 minutes on Colab CPU
print(f'Building embeddings for {len(final_chunks)} chunks...')
vscode_embeddings = []

for chunk in tqdm(final_chunks):
    v = embedding_model.encode(chunk['section'])
    vscode_embeddings.append(v)

vscode_embeddings = np.array(vscode_embeddings)
print(f'Embeddings shape: {vscode_embeddings.shape}')
print('Done!')

Building embeddings for 5735 chunks...


  0%|          | 0/5735 [00:00<?, ?it/s]

Embeddings shape: (5735, 768)
Done!


In [1]:
from minsearch import VectorSearch

vector_index = VectorSearch()
vector_index.fit(vscode_embeddings, final_chunks)
print('Vector index built!')

ModuleNotFoundError: No module named 'minsearch'

In [18]:
# Test vector search with a semantically rephrased query
# (different words, same meaning — where vector search shines)
query = 'I want to remap keys and change hotkeys in my editor'
q_vec = embedding_model.encode(query)
vector_results = vector_index.search(q_vec, num_results=3)

print(f'Query: "{query}"')
print(f'Vector search results: {len(vector_results)}')
print()
for i, r in enumerate(vector_results):
    print(f'[{i+1}] {r.get("title", "No title")} | {r.get("filename", "").split("/")[-1]}')
    print(f'     {r["section"][:200]}')
    print()

Query: "I want to remap keys and change hotkeys in my editor"
Vector search results: 3

[1] No title | v1_8.md
     ## Keyboard shortcuts

### Key binding command arguments

We added support to invoke commands with arguments to the `keybindings.json` configuration file. This is useful if you often perform the same 

[2] No title | keybindings.md
     ## Keyboard Shortcuts editor

VS Code provides a rich keyboard shortcut editing experience with the Keyboard Shortcuts editor. The editor lists all available commands with and without keyboard shortcu

[3] No title | tips-and-tricks.md
     ## Editing hacks

Here is a selection of common features for editing code. If you're more familiar with the keyboard shortcuts for another editor, consider installing a [keymap extension](https://mark



## Part 3: Hybrid Search (Text + Vector combined)

In [19]:
def text_search(query, n=5):
    return text_index.search(query, num_results=n)

def vector_search(query, n=5):
    q_vec = embedding_model.encode(query)
    return vector_index.search(q_vec, num_results=n)

def hybrid_search(query, n=5):
    """Combine text and vector search, deduplicate by filename + section start."""
    text_results   = text_search(query, n)
    vector_results = vector_search(query, n)

    seen = set()
    combined = []

    for r in text_results + vector_results:
        # Use filename + first 60 chars of section as unique key
        key = r.get('filename', '') + r.get('section', '')[:60]
        if key not in seen:
            seen.add(key)
            combined.append(r)

    return combined[:n * 2]  # return up to 2x n results after dedup

print('Search functions defined!')

Search functions defined!


In [20]:
# Full comparison across all 3 search types
test_queries = [
    'How do I configure keyboard shortcuts in VS Code?',
    'I want to remap keys and change hotkeys in my editor',
    'How to install extensions?',
    'debugging python code',
    'version control with git'
]

for query in test_queries:
    t = text_search(query, n=3)
    v = vector_search(query, n=3)
    h = hybrid_search(query, n=3)
    print(f'Query : "{query}"')
    print(f'  Text   → {len(t)} results | Top: {t[0].get("title", "?") if t else "none"}')
    print(f'  Vector → {len(v)} results | Top: {v[0].get("title", "?") if v else "none"}')
    print(f'  Hybrid → {len(h)} results | Top: {h[0].get("title", "?") if h else "none"}')
    print()

Query : "How do I configure keyboard shortcuts in VS Code?"
  Text   → 3 results | Top: ?
  Vector → 3 results | Top: ?
  Hybrid → 5 results | Top: ?

Query : "I want to remap keys and change hotkeys in my editor"
  Text   → 3 results | Top: ?
  Vector → 3 results | Top: ?
  Hybrid → 6 results | Top: ?

Query : "How to install extensions?"
  Text   → 3 results | Top: ?
  Vector → 3 results | Top: ?
  Hybrid → 5 results | Top: ?

Query : "debugging python code"
  Text   → 3 results | Top: ?
  Vector → 3 results | Top: ?
  Hybrid → 6 results | Top: ?

Query : "version control with git"
  Text   → 3 results | Top: ?
  Vector → 3 results | Top: ?
  Hybrid → 6 results | Top: ?



## 🏁 Summary & What's Next

In [21]:
print('=== DAYS 1-3 COMPLETE ===')
print()
print(f'✅ Day 1 — Downloaded microsoft/vscode-docs: {len(vscode_docs)} docs')
print(f'✅ Day 2 — Compared chunking strategies:')
print(f'           Sliding window : {len(vscode_sw_chunks)} chunks')
print(f'           Section-based  : {len(vscode_section_chunks)} chunks  ← chosen')
print(f'✅ Day 3 — Built hybrid search on {len(final_chunks)} chunks')
print(f'           Text index    : keyword/lexical search')
print(f'           Vector index  : semantic/embedding search')
print(f'           Hybrid search : combines both + deduplication')
print()
print('🔜 Day 4 — Build a conversational agent that uses this search!')

=== DAYS 1-3 COMPLETE ===

✅ Day 1 — Downloaded microsoft/vscode-docs: 779 docs
✅ Day 2 — Compared chunking strategies:
           Sliding window : 11770 chunks
           Section-based  : 5735 chunks  ← chosen
✅ Day 3 — Built hybrid search on 5735 chunks
           Text index    : keyword/lexical search
           Vector index  : semantic/embedding search
           Hybrid search : combines both + deduplication

🔜 Day 4 — Build a conversational agent that uses this search!
